# Vision 模型预处理用户拍照（检测 + 分割 + 抠图）\n\n按顺序 Run All：\n1) 加载 `vision/last_ckpt.pth`\n2) 对输入图片做检测+分割\n3) 将抠图结果按 slot 保存到目标衣橱目录（可直接用于后续搭配）\n

In [1]:
from pathlib import Path
import importlib.util
import math
import shutil

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import functional as F

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "polyvore_route_a.py").exists():
            return p
    raise FileNotFoundError("polyvore_route_a.py not found")

project_root = find_project_root()
vision_loader_path = project_root / "vision" / "load model and weight.py"
vision_ckpt_path = project_root / "vision" / "last_ckpt.pth"

print("project_root=", project_root)
print("vision_loader_path=", vision_loader_path, "exists=", vision_loader_path.exists())
print("vision_ckpt_path=", vision_ckpt_path, "exists=", vision_ckpt_path.exists())


project_root= D:\Projects\JupyterProject\COMP7065_GROUP
vision_loader_path= D:\Projects\JupyterProject\COMP7065_GROUP\vision\load model and weight.py exists= True
vision_ckpt_path= D:\Projects\JupyterProject\COMP7065_GROUP\vision\last_ckpt.pth exists= True


In [2]:
# === 可改参数 ===
NUM_CLASSES = 14
SCORE_THRESHOLD = 0.55
MASK_THRESHOLD = 0.50
PADDING_RATIO = 0.06
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_KIND = "clothes"

# 输入用户拍照目录（可替换）
INPUT_DIR = project_root / "user_upload_raw"

# 若 INPUT_DIR 为空，自动用 demo 数据做演示
DEMO_FALLBACK_DIR = project_root / "user_closet_demo" / "top"

# 输出目录：处理后按 slot 落盘，可直接作为衣橱输入
OUTPUT_CLOSET_DIR = project_root / "user_closet_processed"

RESET_OUTPUT_DIR = False

from polyvore_route_a import SLOT_ORDER
ALLOWED_SLOTS = list(SLOT_ORDER)

# DeepFashion2 常见类别映射（如果你的训练标签不同，请在这里改）
CLASS_ID_TO_NAME = {
    1: "short_sleeve_top",
    2: "long_sleeve_top",
    3: "short_sleeve_outwear",
    4: "long_sleeve_outwear",
    5: "vest",
    6: "sling",
    7: "shorts",
    8: "trousers",
    9: "skirt",
    10: "short_sleeve_dress",
    11: "long_sleeve_dress",
    12: "vest_dress",
    13: "sling_dress",
}

# 模型类别 -> 你们项目 slot 映射
CLASS_NAME_TO_SLOT = {
    "short_sleeve_top": "top",
    "long_sleeve_top": "top",
    "vest": "top",
    "sling": "top",
    "short_sleeve_outwear": "outwear",
    "long_sleeve_outwear": "outwear",
    "shorts": "pants",
    "trousers": "pants",
    "skirt": "skirt",
    "short_sleeve_dress": "dress",
    "long_sleeve_dress": "dress",
    "vest_dress": "dress",
    "sling_dress": "dress",
}

print("DEVICE=", DEVICE)
print("OUTPUT_CLOSET_DIR=", OUTPUT_CLOSET_DIR)


DEVICE= cuda
OUTPUT_CLOSET_DIR= D:\Projects\JupyterProject\COMP7065_GROUP\user_closet_processed


In [3]:
spec = importlib.util.spec_from_file_location("vision_loader", vision_loader_path)
vision_loader = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(vision_loader)

model = None

def ensure_vision_model_loaded():
    global model
    if model is None:
        model = vision_loader.load_model_weights(str(vision_ckpt_path), num_classes=NUM_CLASSES, device=DEVICE)
    return model

print("vision_loader_ready=OK")


vision_loader_ready=OK


In [4]:
import io
from datetime import datetime

def choose_main_instance(output: dict, score_threshold: float) -> int | None:
    scores = output["scores"].detach().cpu()
    boxes = output["boxes"].detach().cpu()
    if scores.numel() == 0:
        return None

    valid = torch.where(scores >= score_threshold)[0]
    if valid.numel() == 0:
        valid = torch.tensor([int(torch.argmax(scores).item())])

    # 兼顾置信度和面积，优先选择“最像主衣物”的实例
    best_idx = None
    best_value = -1.0
    for i in valid.tolist():
        x1, y1, x2, y2 = boxes[i].tolist()
        area = max(1.0, (x2 - x1) * (y2 - y1))
        value = float(scores[i]) * math.sqrt(area)
        if value > best_value:
            best_value = value
            best_idx = i
    return int(best_idx) if best_idx is not None else None

def mask_to_cropped_white_bg(img: Image.Image, mask_2d: np.ndarray, padding_ratio: float) -> Image.Image:
    arr = np.array(img)
    h, w = arr.shape[:2]

    ys, xs = np.where(mask_2d)
    if len(xs) == 0 or len(ys) == 0:
        return img.copy()

    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    pad_x = int((x2 - x1 + 1) * padding_ratio)
    pad_y = int((y2 - y1 + 1) * padding_ratio)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w - 1, x2 + pad_x)
    y2 = min(h - 1, y2 + pad_y)

    white = np.full_like(arr, 255)
    white[mask_2d] = arr[mask_2d]
    crop = white[y1 : y2 + 1, x1 : x2 + 1]
    return Image.fromarray(crop)

def estimate_background_rgb(img: Image.Image) -> np.ndarray:
    arr = np.array(img)
    h, w = arr.shape[:2]
    k = max(1, int(min(h, w) * 0.08))
    corners = np.concatenate(
        [
            arr[0:k, 0:k].reshape(-1, 3),
            arr[0:k, w - k : w].reshape(-1, 3),
            arr[h - k : h, 0:k].reshape(-1, 3),
            arr[h - k : h, w - k : w].reshape(-1, 3),
        ],
        axis=0,
    )
    return np.median(corners, axis=0)

def naive_foreground_mask(img: Image.Image) -> np.ndarray:
    arr = np.array(img).astype(np.float32)
    bg = estimate_background_rgb(img).astype(np.float32)
    diff = np.linalg.norm(arr - bg[None, None, :], axis=2)
    thresh = max(15.0, float(np.percentile(diff, 80)))
    mask = diff >= thresh
    return mask

def crop_by_mask_bbox(img: Image.Image, mask_2d: np.ndarray, padding_ratio: float) -> Image.Image:
    arr = np.array(img)
    h, w = arr.shape[:2]
    ys, xs = np.where(mask_2d)
    if len(xs) == 0 or len(ys) == 0:
        return img.copy()

    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    pad_x = int((x2 - x1 + 1) * padding_ratio)
    pad_y = int((y2 - y1 + 1) * padding_ratio)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w - 1, x2 + pad_x)
    y2 = min(h - 1, y2 + pad_y)

    crop = arr[y1 : y2 + 1, x1 : x2 + 1]
    return Image.fromarray(crop)

def clean_accessory_image(img: Image.Image) -> Image.Image:
    mask = naive_foreground_mask(img)
    cropped = crop_by_mask_bbox(img, mask, PADDING_RATIO)
    cropped_mask = naive_foreground_mask(cropped)
    arr = np.array(cropped)
    white = np.full_like(arr, 255)
    white[cropped_mask] = arr[cropped_mask]
    return Image.fromarray(white)

def infer_clothes(img: Image.Image) -> dict:
    _ = ensure_vision_model_loaded()
    x = F.to_tensor(img).to(DEVICE)
    with torch.no_grad():
        output = model([x])[0]

    idx = choose_main_instance(output, SCORE_THRESHOLD)
    if idx is None:
        return {"ok": False, "reason": "no_detection", "img": img}

    label_id = int(output["labels"][idx].detach().cpu().item())
    score = float(output["scores"][idx].detach().cpu().item())
    name = CLASS_ID_TO_NAME.get(label_id, f"class_{label_id}")
    slot = CLASS_NAME_TO_SLOT.get(name, "unknown")

    mask = output["masks"][idx, 0].detach().cpu().numpy() >= MASK_THRESHOLD
    clean = mask_to_cropped_white_bg(img, mask, PADDING_RATIO)
    return {
        "ok": True,
        "img": img,
        "clean": clean,
        "label_id": label_id,
        "label_name": name,
        "slot": slot,
        "score": score,
    }


In [5]:
!pip install ipywidgets

In [6]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as e:
    raise RuntimeError("需要安装 ipywidgets 才能显示上传界面：pip install ipywidgets") from e

if RESET_OUTPUT_DIR and OUTPUT_CLOSET_DIR.exists():
    shutil.rmtree(OUTPUT_CLOSET_DIR)
OUTPUT_CLOSET_DIR.mkdir(parents=True, exist_ok=True)

kind_widget = widgets.ToggleButtons(
    options=[("衣服", "clothes"), ("配饰", "accessory")],
    value=None,
    description="类型",
)

upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="上传")
save_mode_widget = widgets.ToggleButtons(options=[("原图", "raw"), ("抠图", "clean")], value="clean", description="保存")
CLOTH_SLOT_SET = {"top", "outwear", "dress", "jumpsuit", "pants", "skirt", "legwear"}
CLOTH_SLOTS = [s for s in ALLOWED_SLOTS if s in CLOTH_SLOT_SET]
ACCESSORY_SLOTS = [s for s in ALLOWED_SLOTS if s not in CLOTH_SLOT_SET]

def _slots_for_kind(kind: str) -> list[str]:
    return CLOTH_SLOTS if kind == "clothes" else ACCESSORY_SLOTS

slot_widget = widgets.Dropdown(options=[], value=None, description="slot")
save_btn = widgets.Button(description="保存到衣橱", button_style="success")
clear_btn = widgets.Button(description="清空输出", button_style="warning")
out = widgets.Output()

state = {"raw": None, "clean": None, "src_name": None, "info": None}

def _pick_default_slot_for_accessory() -> str:
    if "bag" in ACCESSORY_SLOTS:
        return "bag"
    return ACCESSORY_SLOTS[0] if ACCESSORY_SLOTS else (ALLOWED_SLOTS[0] if ALLOWED_SLOTS else "")

def _set_visible(w: widgets.Widget, visible: bool) -> None:
    w.layout.display = None if visible else "none"

def _reset_after_kind_change(kind: str) -> None:
    state.update({"raw": None, "clean": None, "src_name": None, "info": None})
    slot_widget.options = _slots_for_kind(kind)
    if kind == "accessory":
        slot_widget.value = _pick_default_slot_for_accessory()
    else:
        slots = list(slot_widget.options)
        slot_widget.value = slots[0] if slots else None

    upload_widget.value = {} if isinstance(upload_widget.value, dict) else ()
    _set_visible(upload_row, True)
    _set_visible(slot_row, False)
    _set_visible(save_row, False)
    _render()

def _on_kind_change(change) -> None:
    kind = change.get("new")
    if kind not in {"clothes", "accessory"}:
        return
    _reset_after_kind_change(kind)

def _unique_output_path(slot: str, src_name: str) -> Path:
    slot_dir = OUTPUT_CLOSET_DIR / slot
    slot_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(src_name).stem
    base = slot_dir / f"{stem}_clean.png"
    if not base.exists():
        return base
    suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    alt = slot_dir / f"{stem}_clean_{suffix}.png"
    i = 2
    while alt.exists():
        alt = slot_dir / f"{stem}_clean_{suffix}_{i}.png"
        i += 1
    return alt

def _unique_output_path_raw(slot: str, src_name: str) -> Path:
    slot_dir = OUTPUT_CLOSET_DIR / slot
    slot_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(src_name).stem
    base = slot_dir / f"{stem}_raw.png"
    if not base.exists():
        return base
    suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    alt = slot_dir / f"{stem}_raw_{suffix}.png"
    i = 2
    while alt.exists():
        alt = slot_dir / f"{stem}_raw_{suffix}_{i}.png"
        i += 1
    return alt

def _render() -> None:
    with out:
        clear_output(wait=True)
        if state["raw"] is None:
            print("先选择类型（衣服/配饰），再上传图片。")
            print("输出目录：", OUTPUT_CLOSET_DIR)
            return

        raw = state["raw"]
        clean = state["clean"]
        info = state["info"] or {}
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(raw)
        axes[0].set_title(f"raw: {state['src_name']}")
        axes[0].axis("off")
        axes[1].imshow(clean)
        title = f"clean"
        if info.get("label_name") is not None:
            score = info.get("score")
            score_txt = "N/A" if score is None else f"{float(score):.3f}"
            title += f" | vision={info.get('label_name')} score={score_txt}"
        axes[1].set_title(title)
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

def _get_upload_payload() -> tuple[str, bytes] | None:
    v = upload_widget.value
    if not v:
        return None

    # ipywidgets 7: dict(filename -> {metadata:{name}, content:bytes, ...})
    if isinstance(v, dict):
        key = next(iter(v.keys()))
        fileobj = v[key]
        name = fileobj.get("metadata", {}).get("name") or key or "upload.png"
        data = fileobj.get("content")
        if data is None:
            return None
        return str(name), data

    # ipywidgets 8: tuple of dicts: {name, type, size, content, ...}
    if isinstance(v, (tuple, list)):
        fileobj = v[0]
        if isinstance(fileobj, dict):
            name = fileobj.get("name") or "upload.png"
            data = fileobj.get("content")
            if data is None:
                return None
            return str(name), data

    return None

def _try_clear_upload() -> None:
    try:
        upload_widget.value.clear()
        upload_widget._counter = 0
        return
    except Exception:
        pass
    try:
        upload_widget.value = ()
    except Exception:
        pass

def _process_upload(_change=None) -> None:
    payload = _get_upload_payload()
    if payload is None:
        return

    src_name, data = payload
    try:
        img = Image.open(io.BytesIO(data)).convert("RGB")
        kind = kind_widget.value
        if kind not in {"clothes", "accessory"}:
            with out:
                print("请先选择类型")
            _try_clear_upload()
            return

        if kind == "clothes":
            r = infer_clothes(img)
            if not r.get("ok"):
                state.update({"raw": img, "clean": img, "src_name": src_name, "info": {"error": r.get('reason')}})
                _render()
                _try_clear_upload()
                return
            clean = r["clean"]
            suggested = r.get("slot")
            if suggested in ALLOWED_SLOTS:
                slot_widget.value = suggested
            state.update({"raw": img, "clean": clean, "src_name": src_name, "info": {"label_name": r.get('label_name'), "score": r.get('score')}})
        else:
            clean = clean_accessory_image(img)
            if slot_widget.value not in ALLOWED_SLOTS:
                slot_widget.value = _pick_default_slot_for_accessory()
            state.update({"raw": img, "clean": clean, "src_name": src_name, "info": {}})

        _set_visible(slot_row, True)
        _set_visible(save_row, True)
        save_mode_widget.value = "clean"
        _render()
    except Exception as e:
        with out:
            print("处理上传图片失败：", repr(e))
    finally:
        _try_clear_upload()

def _save(_btn) -> None:
    if state["clean"] is None:
        _process_upload()
    if state["clean"] is None:
        with out:
            print("请先上传图片")
        return
    slot = slot_widget.value
    if slot not in ALLOWED_SLOTS:
        raise ValueError(f"非法 slot={slot!r}")
    src_name = state["src_name"] or "upload.png"
    if save_mode_widget.value == "raw":
        dst = _unique_output_path_raw(slot, src_name)
        state["raw"].save(dst)
    else:
        dst = _unique_output_path(slot, src_name)
        state["clean"].save(dst)
    with out:
        print("saved=", dst)

def _clear(_btn) -> None:
    if OUTPUT_CLOSET_DIR.exists():
        shutil.rmtree(OUTPUT_CLOSET_DIR)
    OUTPUT_CLOSET_DIR.mkdir(parents=True, exist_ok=True)
    state.update({"raw": None, "clean": None, "src_name": None, "info": None})
    _set_visible(slot_row, False)
    _set_visible(save_row, False)
    _render()

upload_row = widgets.HBox([upload_widget])
slot_row = widgets.HBox([slot_widget])
save_row = widgets.HBox([save_mode_widget, save_btn, clear_btn])

_set_visible(upload_row, False)
_set_visible(slot_row, False)
_set_visible(save_row, False)

kind_widget.observe(_on_kind_change, names="value")
upload_widget.observe(_process_upload, names="value")
save_btn.on_click(_save)
clear_btn.on_click(_clear)

display(widgets.VBox([widgets.HBox([kind_widget]), upload_row, slot_row, out, save_row]))
_render()


In [7]:
from collections import Counter

if not OUTPUT_CLOSET_DIR.exists():
    print("output_dir_not_found=", OUTPUT_CLOSET_DIR)
else:
    slot_counter = Counter()
    for d in OUTPUT_CLOSET_DIR.iterdir():
        if not d.is_dir():
            continue
        n = len(list(d.glob('*.png')))
        if n:
            slot_counter[d.name] += n
    print("slot_counts=", dict(slot_counter))


slot_counts= {'pants': 1}


In [8]:
print("请使用上方上传界面预览与保存；本 cell 不再重复展示。")


请使用上方上传界面预览与保存；本 cell 不再重复展示。


## 下一步接入搭配推荐\n处理完成后，`OUTPUT_CLOSET_DIR` 就是可直接用于推荐的衣橱根目录。\n你可以在 `closet_recommend.ipynb` 里把 `closet_root` 指向这个目录（例如 `project_root / "user_closet_processed"`）。